In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from config import DATA_DIR



In [2]:
df = pd.read_csv(f"{DATA_DIR}/df_raw.csv", index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index, utc=True)
print(df.shape)
df = df["2024-01-01":]
print(df.shape)
df.head()

(34320, 7)
(16800, 7)


,net_load,DA_load,actual_load,DA_solar,DA_wind,actual_solar,actual_wind
2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-01-01 01:00:00+00:00,36625.0,51850.0,50992.0,0.0,14601.58,0.0,14367.0
2024-01-01 02:00:00+00:00,34794.0,49250.0,48187.0,0.0,14179.37,0.0,13393.0
2024-01-01 03:00:00+00:00,37872.0,47050.0,46479.0,0.0,13754.28,0.0,8607.0
2024-01-01 04:00:00+00:00,38037.0,46150.0,46285.0,0.0,13076.23,0.0,8248.0


In [3]:
# create input feature
df["net_load_24"] = df["net_load"].shift(24)
df["net_load_25"] = df["net_load"].shift(25)
df["net_load_26"] = df["net_load"].shift(26)

df["DA_renewable"] = df["DA_solar"] + df["DA_wind"]
df["DA_renewable_1"] = df["DA_renewable"].shift(1)
df["DA_renewable_2"] = df["DA_renewable"].shift(2)
df["DA_renewable_3"] = df["DA_renewable"].shift(3)

df["DA_load_1"] = df["DA_load"].shift(1)
df["DA_load_2"] = df["DA_load"].shift(2)
df["DA_load_3"] = df["DA_load"].shift(3)

hours = df.index.hour
df["hour_sin"] = np.sin(2 * np.pi * hours / 24)
df["hour_cos"] = np.cos(2 * np.pi * hours / 24)

input_features = [
    "net_load_24",
    "net_load_25",
    "net_load_26",
    "DA_renewable",
    "DA_renewable_1",
    "DA_renewable_2",
    "DA_renewable_3",
    "DA_load",
    "DA_load_1",
    "DA_load_2",
    "DA_load_3",
    "hour_sin",
    "hour_cos",
]
%store input_features

Stored 'input_features' (list)


In [4]:
# Enregistrer le dataframe en csv
df.to_csv(f'{DATA_DIR}/df.csv', index=True)

df.head()

,net_load,DA_load,actual_load,DA_solar,DA_wind,actual_solar,actual_wind,net_load_24,net_load_25,net_load_26,DA_renewable,DA_renewable_1,DA_renewable_2,DA_renewable_3,DA_load_1,DA_load_2,DA_load_3,hour_sin,hour_cos
2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000
2024-01-01 01:00:00+00:00,36625.0,51850.0,50992.0,0.0,14601.58,0.0,14367.0,NaN,NaN,NaN,14601.58,NaN,NaN,NaN,NaN,NaN,NaN,0.258819,0.965926
2024-01-01 02:00:00+00:00,34794.0,49250.0,48187.0,0.0,14179.37,0.0,13393.0,NaN,NaN,NaN,14179.37,14601.58,NaN,NaN,51850.0,NaN,NaN,0.500000,0.866025
2024-01-01 03:00:00+00:00,37872.0,47050.0,46479.0,0.0,13754.28,0.0,8607.0,NaN,NaN,NaN,13754.28,14179.37,14601.58,NaN,49250.0,51850.0,NaN,0.707107,0.707107
2024-01-01 04:00:00+00:00,38037.0,46150.0,46285.0,0.0,13076.23,0.0,8248.0,NaN,NaN,NaN,13076.23,13754.28,14179.37,14601.58,47050.0,49250.0,51850.0,0.866025,0.500000
